# 06 — Sensibilidade da recomendação

Cada cenário refaz a cadeia a partir do ponto em que o parâmetro entra (`notebooks/sensibilidade.py`;
executado com a variável de ambiente `SENS`) e grava em `resultados/sensibilidade/<cenário>/`:

| Cenário | O que muda | Refaz |
|---|---|---|
| CT1350 / CT1550 | soma térmica da semeadura à maturação: 1450 ± 100 °C dia | 02 → 03 → 05 |
| janela_cedo | pesos 0,4 / 0,3 / 0,2 / 0,1 nas datas 15/01, 05/02, 25/02, 15/03 (semeadura concentrada logo após a soja) | 03 → 05 |
| janela_7datas | as 7 datas da janela do ZARC (15/01 a 15/03), pesos iguais | 03 → 05 |

Os cenários econômicos (custo total, cobertura de 70%, ZARC Grupo II, 10 ou 30 dias até a venda) não
mexem no modelo e são calculados no próprio notebook 05 (`sensibilidade_decisao.csv`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from sensibilidade import CENARIOS, RESULTADOS

RAIZ = Path.cwd().parent
FIGURAS = RAIZ / "figuras"
ORDEM_DATAS = [f"{d:02d}/{m:02d}" for m in (1, 2, 3) for d in (5, 15, 25)]
FINAL = "L_ensemble_E_SVR"
PASTAS = {"base": RESULTADOS} | {c: RESULTADOS / "sensibilidade" / c for c in CENARIOS}
faltam = [c for c, p in PASTAS.items() if not (p / "melhor_data_semeadura_pr.csv").exists()]
print("Cenários sem resultado:", faltam or "nenhum")

## 1. Modelo, ciclo e decisão por cenário

In [ ]:
base_dec = pd.read_csv(RESULTADOS / "melhor_data_semeadura_pr.csv", index_col=0)
linhas, curvas = [], {}
for nome, pasta in PASTAS.items():
    val = pd.read_csv(pasta / "validacao_loyo_modelos.csv", index_col=0) if (pasta / "validacao_loyo_modelos.csv").exists() \
        else pd.read_csv(RESULTADOS / "validacao_loyo_modelos.csv", index_col=0)
    dec = pd.read_csv(pasta / "melhor_data_semeadura_pr.csv", index_col=0)
    risco = pd.read_csv(pasta / "risco_economico_municipio_data.csv")
    sim_arq = pasta / "simulacao_mza_pr.parquet" if (pasta / "simulacao_mza_pr.parquet").exists() else RESULTADOS / "simulacao_mza_pr.parquet"
    ciclo = pd.read_parquet(sim_arq, columns=["semeadura", "ciclo_dias"]).groupby("semeadura")["ciclo_dias"].mean()
    melhor = risco.merge(dec["melhor_data_zarc"].rename("semeadura").reset_index(), on=["cod_ibge", "semeadura"])
    curvas[nome] = risco.groupby("semeadura")["CE_rr2"].mean().loc[ORDEM_DATAS]
    ok = dec["melhor_data_zarc"].notna() & base_dec["melhor_data_zarc"].reindex(dec.index).notna()
    linhas.append({
        "cenario": nome,
        "R2_municipal": val.loc[FINAL, "R2"], "R2_anual": val.loc[FINAL, "R2_anual_estadual"],
        "RMSE_kg_ha": val.loc[FINAL, "RMSE_kg_ha"],
        "ciclo_05jan_d": ciclo["05/01"], "ciclo_15mar_d": ciclo["15/03"],
        "mesma_data_base_%": 100 * (dec.loc[ok, "melhor_data_zarc"] == base_dec.loc[dec.index[ok], "melhor_data_zarc"]).mean(),
        "melhor_05jan_n": int((dec["melhor_data_zarc"] == "05/01").sum()),
        "melhor_15mar_n": int((dec["melhor_data_zarc"] == "15/03").sum()),
        "CE_melhor_data": melhor["CE_rr2"].mean(), "p_prejuizo_melhor_data": melhor["p_prejuizo"].mean(),
        "valor_seguro_medio": risco["valor_seguro"].mean(),
    })
tab = pd.DataFrame(linhas).set_index("cenario")
tab.to_csv(RESULTADOS / "sensibilidade_cenarios.csv")
tab.round(3)

## 2. Cenários econômicos (notebook 05)

In [ ]:
pd.read_csv(RESULTADOS / "sensibilidade_decisao.csv").round(2)

## 3. Equivalente certo por data de semeadura

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for nome, c in curvas.items():
    ax.plot(range(len(ORDEM_DATAS)), c.values, "k-o" if nome == "base" else "--.", lw=2 if nome == "base" else 1, label=nome)
ax.set_xticks(range(len(ORDEM_DATAS)), ORDEM_DATAS)
ax.set(xlabel="Data de semeadura", ylabel=r"Equivalente certo médio (R\$ ha$^{-1}$, rr = 2)")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
fig.savefig(FIGURAS / "fig_sensibilidade_ce.png", dpi=300)